Intermediate: IP flows (i.e. WIPO non-class IP Indicators)
==========================================================


In [1]:
import pandas as pd
from pathlib import Path
import sys

In [2]:
# Setup Paths
current_dir = Path.cwd()
staging_path = (current_dir.parent.parent / "data" / "staging" / "wipo" / "ip_indicators").resolve()
intermediate_path = (current_dir.parent.parent / "data" / "intermediate").resolve()
intermediate_path.mkdir(parents=True, exist_ok=True)
output_file = intermediate_path / "int_ip_flows.parquet"

print(f"Reading from: {staging_path}")
print(f"Exporting to: {output_file}\n")

Reading from: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\wipo\ip_indicators
Exporting to: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\intermediate\int_ip_flows.parquet



In [3]:
target_files = [
    "stg_wipo__ID1a.parquet",
    "stg_wipo__ID1b.parquet",
    "stg_wipo__ID2a.parquet",
    "stg_wipo__ID2b.parquet",
    "stg_wipo__ID7.parquet",
    "stg_wipo__PA1a.parquet",
    "stg_wipo__PA1b.parquet",
    "stg_wipo__PA2a.parquet",
    "stg_wipo__PA2b.parquet",
    "stg_wipo__PA3.parquet",
    "stg_wipo__TM1a.parquet",
    "stg_wipo__TM1b.parquet",
    "stg_wipo__TM2a.parquet",
    "stg_wipo__TM2b.parquet",
    "stg_wipo__TM7.parquet"
]

In [4]:
dfs_to_combine = []

In [5]:
for file_name in target_files:
    file_path = staging_path / file_name
    
    if not file_path.exists():
        print(f"[MISSING] {file_name}")
        continue
        
    try:
        # Read
        df = pd.read_parquet(file_path)
        
        # Standardize headers (ensure lowercase for matching)
        df.columns = df.columns.astype(str).str.strip().str.lower()
        
        # Add Indicator (from filename)
        df['indicator'] = file_name.replace("stg_wipo__", "").replace(".parquet", "")

        # Melt (Wide -> Long)
        # We hold 'office', 'origin', 'indicator' constant and melt the rest (years)
        df_melted = df.melt(
            id_vars=['office', 'origin', 'indicator'], 
            var_name='year', 
            value_name='count'
        )
        
        # Reorder Columns immediately: year, origin, office, indicator, count
        df_melted = df_melted[['year', 'origin', 'office', 'indicator', 'count']]
        
        dfs_to_combine.append(df_melted)
        print(f"Processed: {file_name}")

    except Exception as e:
        print(f"[ERROR] {file_name}: {e}")

Processed: stg_wipo__ID1a.parquet
Processed: stg_wipo__ID1b.parquet
Processed: stg_wipo__ID2a.parquet
Processed: stg_wipo__ID2b.parquet
Processed: stg_wipo__ID7.parquet
Processed: stg_wipo__PA1a.parquet
Processed: stg_wipo__PA1b.parquet
Processed: stg_wipo__PA2a.parquet
Processed: stg_wipo__PA2b.parquet
Processed: stg_wipo__PA3.parquet
Processed: stg_wipo__TM1a.parquet
Processed: stg_wipo__TM1b.parquet
Processed: stg_wipo__TM2a.parquet
Processed: stg_wipo__TM2b.parquet
Processed: stg_wipo__TM7.parquet


In [6]:
# Save
if dfs_to_combine:
    final_df = pd.concat(dfs_to_combine, ignore_index=True)
    
    # Export
    final_df.to_parquet(output_file, index=False)
    print(f"\nCompleted. Saved {len(final_df)} rows to: {output_file}")
    print(final_df.head())
else:
    print("\nNo data processed.")


Completed. Saved 4474575 rows to: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\intermediate\int_ip_flows.parquet
   year        origin                                      office indicator  \
0  1980     Australia  African Intellectual Property Organization      ID1a   
1  1980       Belgium  African Intellectual Property Organization      ID1a   
2  1980  Burkina Faso  African Intellectual Property Organization      ID1a   
3  1980         Benin  African Intellectual Property Organization      ID1a   
4  1980        Brazil  African Intellectual Property Organization      ID1a   

   count  
0    NaN  
1    NaN  
2    NaN  
3    NaN  
4    NaN  
